# 03. Feature Engineering & Signal Provenance Auditing

## Methodological Framing
Features must be strictly constructed from observation window metrics ($t \le T$). We record explicit provenance metadata for every signal and audit all candidate features through our 7-tier leakage prevention protocol.

### Objectives:
1. Transform raw search performance metrics into normalized, robust indicators.
2. Enforce zero-denominator protection on CTR and momentum ratios.
3. Execute pre-training leakage audit to guarantee no BLOCKED features enter modeling.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

root_dir = Path.cwd().parent if Path.cwd().name == "work" else Path.cwd()
sys.path.insert(0, str(root_dir))

from src.config import load_config
from src.data import load_dataset
from src.features import build_feature_table
from src.leakage import LeakageAuditor

config = load_config()
auditor = LeakageAuditor(config.schema_mapping.get("features", []))

## Step 1: Feature Transformation & Provenance Logging

In [ ]:
try:
    df = load_dataset(config.raw_data_path, config)
    X, feature_names = build_feature_table(df, config.schema_mapping.get("features", []))
    print(f"Engineered {len(feature_names)} features:")
    for f in feature_names:
        meta = auditor.audit_feature(f)
        print(f" - {f:<30} | Status: {meta.leakage_status:<8} | Rule: {meta.observation_time_rule}")
except FileNotFoundError:
    print("[STATUS: AWAITING REAL DATA EXECUTION] - Dataset connection required.")

## Step 2: Leakage Gate Verification
Testing that intentional leakers are blocked:

In [ ]:
for test_col in ["trend_direction", "future_clicks", "recommended_action"]:
    res = auditor.audit_feature(test_col)
    print(f"Auditing '{test_col}': Status={res.leakage_status}, Reason={res.notes}")
    assert res.leakage_status == "BLOCKED", f"Expected BLOCKED for {test_col}"